In [ ]:
# lib import
import os
import torch
import numpy as np
from tqdm import tqdm
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

# setup
ds_size = 100000
source_ds_cache_dir = os.path.join(os.getcwd(), "data", "hf-source")

embed_models = [
    "all-MiniLM-L6-v2",
    "sentence-transformers/all-mpnet-base-v2",
    "nomic-ai/nomic-embed-text-v1.5"
    ]
embed_ds_dirnames = []
for name in embed_models:
    embed_ds_dirnames.append(name.split("/")[-1])

ds_pos_results = []

device = "cuda" if torch.cuda.is_available() else "cpu"
print("running on " + device)

# Acquiring the Source Dataset
Using Huggingface's Datasets we can easily load the wikipedia (20231101.en) dataset. Since we are focusing on encyclopedic data, this will serve as the base for testing the dataset generation pipeline.

In [ ]:
source_ds_cache_dir = os.path.join(os.getcwd(), "data", "hf-source")
ds = load_dataset("wikimedia/wikipedia", "20231101.en", cache_dir=source_ds_cache_dir)

In [ ]:
# verification (optional)
print(f"Dataset length: {len(ds['train'])}")
print(ds['train'][0])
print(ds['train'][-1])

In [ ]:
# create subset for dev
ds_shuffled = ds.shuffle(seed=97)
if ds_size > 0:
    ds_subset = ds_shuffled['train'].select(range(ds_size))
else:
    ds_subset = ds_shuffled['train']
    
print(f"Test subset length: {len(ds_subset)}")
print(f"Sample entry: {ds_subset[0]['title']}")

# Embedding Generation

In [ ]:
embed_ds = []
print("Preparing texts...")
texts = ds_subset['text']

for index, model_name in enumerate(embed_models):
    ds_embed_dir = os.path.join(os.getcwd(), "data", "ds_embed",  embed_ds_dirnames[index], str(ds_size))
    embed_ds.append(ds_subset)

    print("Loading model...")
    model = SentenceTransformer(model_name, trust_remote_code=True, device=device)
    tokenizer = model.tokenizer
    print("Tokenizer loaded:")
    print(tokenizer)
    
    print("Generating tokens...")
    batch_encoding = tokenizer(
        texts,
        padding=True,
        truncation=True, 
        return_tensors=None,
        add_special_tokens=True
    )
    
    print("Adding raw token ids to embed dataset...")
    embed_ds[index] = embed_ds[index].add_column("token_ids", batch_encoding['input_ids'])

    print("Saving dataset with tokens...")
    embed_ds[index].save_to_disk(ds_embed_dir)

    print("Generating embeddings...")
    batch_encoding = model.encode(texts, show_progress_bar=True, batch_size=1, device=device)
    embed_ds[index] = embed_ds[index].add_column("embeddings", batch_encoding.tolist())

    print("Saving dataset with embeddings...")
    embed_ds[index].save_to_disk(ds_embed_dir)

# Processing


In [ ]:
import numpy as np
from numpy import ndarray
from datasets import Dataset, DatasetDict
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA


def prep(data: Dataset | DatasetDict) -> ndarray:
    # L2-normalization (to euclidean instead of cosine)
    embeddings_array = np.asarray(data, dtype=np.float32)
    embeddings_norm = normalize(embeddings_array, norm="l2", axis=1)

    # PCA red to 50 dims: De-noising and speedup
    pca = PCA(n_components=50, random_state=97, svd_solver="auto", whiten=False)
    data = pca.fit_transform(embeddings_norm)
    return data

In [ ]:

from sklearn.preprocessing import MinMaxScaler

def normalize_coords(embed_reduced, target_range=(-1, 1)):
    scaler = MinMaxScaler(feature_range=target_range)
    return scaler.fit_transform(embed_reduced)

# Pipeline A (UMAP)
Based on [`knoto`](https://github.com/whatphilipcodes/knoto), this pipeline uses `UMAP` to reduce `sBERT` embeddings into two (spatial) dimensions.

In [ ]:
import umap
from datasets import load_from_disk

for dirname in embed_ds_dirnames:
    ds_embed_dir = os.path.join(os.getcwd(), "data", "ds_embed", dirname, str(ds_size))
    ds_pos_dir = os.path.join(os.getcwd(), "data", "ds_pos", dirname, "umap", str(ds_size))

    ds_embed = load_from_disk(ds_embed_dir)
    print(f"Loaded dataset with {len(ds_embed)} entries")

    # collect embeddings
    embeddings_array = prep(ds_embed['embeddings'])
    print(f"Embeddings shape: {embeddings_array.shape}")

    # apply UMAP reduction
    reducer = umap.UMAP(n_components=2, random_state=97, init="pca")
    print("Applying UMAP reduction...")
    embed_reduced = reducer.fit_transform(embeddings_array)
    print(f"2D embeddings shape: {embed_reduced.shape}")

    # apply normalization
    embed_reduced = normalize_coords(embed_reduced)

    # add to ds
    ds_pos = ds_embed.add_column("x", embed_reduced[:, 0].tolist())
    ds_pos = ds_pos.add_column("y", embed_reduced[:, 1].tolist())
    print(f"Final dataset columns: {ds_pos.column_names}")

    # save ds with positions
    ds_pos.save_to_disk(ds_pos_dir)
    ds_pos_results.append(ds_pos_dir)
    print("Dataset with UMAP positions saved successfully!")

# Pipeline B (t-SNE)
based on the `scikit-learn` implementation of `t-SNE`.

In [ ]:
from sklearn.manifold import TSNE
from datasets import load_from_disk

for dirname in embed_ds_dirnames:
    ds_embed_dir = os.path.join(os.getcwd(), "data", "ds_embed", dirname, str(ds_size))
    ds_pos_dir = os.path.join(os.getcwd(), "data", "ds_pos", dirname, "tsne", str(ds_size))

    # load dataset with embeddings (reuse from UMAP pipeline)
    ds_embed = load_from_disk(ds_embed_dir)
    print(f"Loaded dataset with {len(ds_embed)} entries")

    # collect embeddings
    embeddings_array = prep(ds_embed['embeddings'])
    print(f"Embeddings shape: {embeddings_array.shape}")

    # apply t-SNE reduction
    tsne = TSNE(n_components=2, random_state=97, init="pca")
    print("Applying t-SNE reduction...")
    embed_reduced_tsne = tsne.fit_transform(embeddings_array)
    print(f"2D t-SNE embeddings shape: {embed_reduced_tsne.shape}")

    # normalize coords
    embed_reduced_tsne = normalize_coords(embed_reduced_tsne)

    # add to dataset
    ds_pos_tsne = ds_embed.add_column("x", embed_reduced_tsne[:, 0].tolist())
    ds_pos_tsne = ds_pos_tsne.add_column("y", embed_reduced_tsne[:, 1].tolist())
    print(f"Final dataset columns: {ds_pos_tsne.column_names}")

    # save dataset with t-SNE positions
    ds_pos_tsne.save_to_disk(ds_pos_dir)
    ds_pos_results.append(ds_pos_dir)
    print("Dataset with t-SNE positions saved successfully!")

# Visualization

In [ ]:
from datasets import load_from_disk
import matplotlib.pyplot as plt
import seaborn as sns
import os
import random
from datetime import datetime

img_dir = os.path.join(os.getcwd(), "img")
# Ensure img directory exists
os.makedirs(img_dir, exist_ok=True)

for dir in ds_pos_results:
    ds_pos = load_from_disk(dir)

    xs = ds_pos["x"]
    ys = ds_pos["y"]
    labels = ds_pos["title"]

    plt.figure(figsize=(8, 8))  # Square figure size
    
    random.seed(97)
    random_indices = set(random.sample(range(len(xs)), min(24, len(xs))))

    # Plot unlabeled points in blue
    unlabeled_x = [x for i, x in enumerate(xs) if i not in random_indices]
    unlabeled_y = [y for i, y in enumerate(ys) if i not in random_indices]
    
    # Plot labeled points in red
    labeled_x = [x for i, x in enumerate(xs) if i in random_indices]
    labeled_y = [y for i, y in enumerate(ys) if i in random_indices]

    palette = sns.color_palette("Set2")
    sns.scatterplot(x=unlabeled_x, y=unlabeled_y, s=10, alpha=0.7, color=palette[0])
    sns.scatterplot(x=labeled_x, y=labeled_y, s=10, alpha=0.7, color=palette[1])

    for i, (x, y, label) in enumerate(zip(xs, ys, labels)):
        if i in random_indices:
            plt.annotate(label, (x, y), xytext=(5, 5), textcoords='offset points', fontsize=8, bbox=dict(boxstyle="round,pad=0.3", facecolor='white', alpha=1.0, edgecolor='gray'))

    # Set fixed square aspect ratio and limits
    plt.xlim(-1.2, 1.2)
    plt.ylim(-1.2, 1.2)
    
    # Generate filename with timestamp
    timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")
    # Extract directory name parts for filename
    dir_parts = dir.replace(os.getcwd(), "").strip(os.sep).split(os.sep)
    # Create filename: embedding_model_reduction_method_size_timestamp
    filename = f"{dir_parts[2]}_{dir_parts[3]}_{dir_parts[4]}_{timestamp}.pdf"
    filepath = os.path.join(img_dir, filename)
    
    # Save the plot as PDF (vector format, best for LaTeX)
    plt.savefig(filepath, format='pdf', bbox_inches='tight')
    print(f"Saved plot to: {filepath}")
    
    plt.show()